# Curation of Crop Data

## 1. Environment Setup and Configuration

### Test kernel

Just for checking the environment is working.

In [ ]:
txt = "Hello"
print(txt)

### Import Required Libraries

In [ ]:
# For calling data repository URL
import requests
import io

# For manipulating data with Pandas
import pandas as pd

# Ensure the odfpy library is installed
%pip install odfpy

## 2. Plan

- Configure Environment.
- Retrieve weather data from URL.
  - Raw data, with unnecessary metadata removed.
- Download and save as csv for offline use.
- Parse csv into dataframe for wrangling in readiness for analytics.

### Thoughts

- A clean dataframe of the specific data that I want.
- Maybe export to new cleansed CSV file ready for ingestion into (SQLite) database.
- Think the crop data may only be avilable as csv downloads.
- Might think to automate download, but in the first instance, just download and process.
- Crop yield data is available for several crops, over many years, by each region.
- Paired data is for weather at a regional level.

### General Plan

1. Get data from url.
2. Parse into dataframe for manipulation.
  2.1 The dataframe could be output to csv.
3. Format data to useful dataframe for analysis.
4. Conduct exploratory data analysis to identify interesting features for exploration.
5. Analyse interesting feature interactions with statisitical models/ML.
6. Explore interesing interactions with further EDA and ML.

- Direct retrival of data from url
  - Perhaps download raw then process?
  - Alternatively direct manipulation for smaller dataset.
- Ingestion into dataframe.
- Cleansing through manipulation of dataframe.
- Output cleansed CSV for next process in the pipeline.

- ETL
  - Extract
  - Transform
  - Load

- ELT
  - Extract
  - Load
  - Tranform

## 3. Extract Data from Government Online Repository 

- Source data from URL.
- Parse into Pandas DataFrame.
- List sheets in ODF workbook.
- For each sheet in ODF workbook:
  - Extract as dataframe.
  - Export to local storage as CSV.

### Region Labels

Need to consider how the data is going to be aggregated. This needs to match with the aggregation of weather data, so will have to aggregate to the lowest common denominator.

Initial EDA may hihglight regional or country level variation highlighting features for exploration.

#### Crop Data Grouping Lists

These lists show how the data is presented in the source data.

<table>

<tr><th>Whole List</th><th>UK Countries</th><th>UK Only Regions</th></tr>
<tr><td>

|ID|Crop Regions|
|--|------------|
|1|United Kingdom|
|2|England|
|3|North East|
|4|North West and Merseyside|
|5|Yorkshire & The Humber|
|6|East Midlands|
|7|West Midlands|
|8|Eastern|
|9|South East and London|
|10|South West|
|11|Wales|
|12|Scotland|
|13|Northern Ireland|

</td><td>

|ID|Crop Regions|
|--|------------|
|1|United Kingdom|
|2|England|
|11|Wales|
|12|Scotland|
|13|Northern Ireland|

</td><td>

|ID|Crop Regions|
|--|------------|
|3|North East|
|4|North West and Merseyside|
|5|Yorkshire & The Humber|
|6|East Midlands|
|7|West Midlands|
|8|Eastern|
|9|South East and London|
|10|South West|

</td></tr> </table>

### Source data from Government repository

Try following url: <https://assets.publishing.service.gov.uk/media/68e778c1e5f463a62cb98588/UK-cops-webseries-251009a.ods>

Objects:

- label: **buffer_object**, object type: **_io.BytesIO**

In [ ]:
# N.B. Need to ensure odfpy is installed in the environment. This allows the ODS binary to be parsed.

# Creates a ByteIO Class Object

import requests
import io

url = "https://assets.publishing.service.gov.uk/media/68e778c1e5f463a62cb98588/UK-cops-webseries-251009a.ods"
response = requests.get(url)
response.raise_for_status() # This gives the REST request response.

buffer_object = io.BytesIO(response.content)

In [ ]:
type(buffer_object)

### Identify the sheets in the workbook

Objects:
- label: **list_odf_sheets**, object type: **list**
- label: **sheets_list**, object typeL **.csv**


In [ ]:
# https://pandas.pydata.org/docs/dev/reference/api/pandas.ExcelFile.sheet_names.html
# this outputs an ExcelFile Class Object

import pandas as pd

crops_file = pd.ExcelFile(
				path_or_buffer=buffer_object
				,engine="odf")

list_odf_sheets: list = []

for sheet in crops_file.sheet_names:
	list_odf_sheets.append(sheet)

list_odf_sheets

Output list to a .csv using Pandas

In [ ]:
df_odf_list = pd.DataFrame(list_odf_sheets)

df_odf_list.to_csv('data/crops/sheet_list.csv', index=False)

df_odf_list.head()

#### Parse each sheet into separate .csv

For each sheet with suffix 'Region' parse into from **buffer_object** into dataframes and output as local .csv files.

Objects:
- label: **raw_wheat.csv**, object type: **.csv**
- label: **raw_winter_barley.csv**, object type: **.csv**
- label: **raw_spring_barley.csv**, object type: **.csv**
- label: **raw_total_barley.csv**, object type: **.csv**
- label: **raw_oats.csv**, object type: **.csv**
- label: **raw_osr.csv**, object type: **.csv**

Locally saving the file allows decoupling the need for network calls, allowing for faster I/O operations. This also allows me to continue development whilst internet connection was not availalble, e.g. whilst travelling on the train.

Exporting locally to .csv files does come at the cost of storage. This can be mitigated by deleting the files once processing is completed, retaining only the required data for reference. However, the size of data is very small and easily maintained on even the most limited of systems.

In [ ]:
# Define the prefix
prefix = 'Region'

# Ensure that the directory exists before writing files
import os
os.makedirs('data/crops', exist_ok=True)

# Use for loop to list through list of sheet names, parse sheet into dataframe, and output locally as .csv file.
for name in list_odf_sheets:
	if name.startswith(prefix):
		df_sheet = (pd.ExcelFile(path_or_buffer=buffer_object,engine="odf")).parse(name)
		df_sheet.to_csv(f'data/crops/{name}.csv', index=False)

## 4. Clean data

- For each sheet CSV:
  - load into dataframe.
  - crop to new dataframe with relevant rows and columns.
  - Reset column headers and index to years.

### 4.1 Load single crop .csv into DataFrame

In [ ]:
# Will start with the Regional_spring_barley.csv
df_spring_barley = pd.read_csv('data/crops/Regional_spring_barley.csv', header=2)

df_spring_barley.head(26)

### 4.2 Check Columns

In [ ]:
df_spring_barley.columns

The column names need are for the years of the data. The current DataFrame includes blank rows due to the structure of the odf worksheet. Therefore it will be necessary to crop the DataFrame and reset the column headers index.

### 4.3 Crop data to a resized DataFrame

Looking at the original odf download, we can determine the row in which the year column headers can be found. Since all crop sheets have identical structure, we can use the same logic.

There may be progammatic means to achieve this, but given ease with which this can be identified, pragmatism would suggest this is uncessary.

The a new DataFrame can be created by selecting a subset of the existing DataFrame using the iloc operator to specify their index location in the DataFrame. N.B. using the loc operator is used for selecting based on row and column names rather than index position.

First to show rows and columns, can use the iloc index function.

```python
df.iloc[0:10, 0:10] # will show the first 10 rows and first 10 columns.
```
**Reference:** <https://pandas.pydata.org/docs/getting_started/intro_tutorials/03_subset_data.html>

In [ ]:
df_sb_yields = df_spring_barley.iloc[15:28, 0:28]
df_sb_yields.reset_index(drop=True, inplace=True)
df_sb_yields.head(13)

### Transpose data with years as rows

In [ ]:
# ref https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.transpose.html

df_yields_sb_long = df_sb_yields.transpose()

df_yields_sb_long.head()

In [ ]:
df_yields_sb_long.iloc[0]

### Reset column index to correct column index

In [ ]:
# Make the first row, row index 0, the column names.
df_yields_sb_long.columns = df_yields_sb_long.iloc[0]

df_yields_sb_long.head(5)

In [ ]:
# Reset the dataframe to start without the first row that has now become the column names.
df_yields_sb_long = df_yields_sb_long[1:]

df_yields_sb_long.head(5)

In [ ]:
# Clear the column heading name.
df_yields_sb_long.columns.name = None

df_yields_sb_long.head(5)

### Convert data types

In [ ]:
# Check types
df_yields_sb_long.dtypes

In [ ]:
# For each column in the dataframe, convert the data type to float.
for column in df_yields_sb_long.columns:
	df_yields_sb_long[column] = df_yields_sb_long[column].astype(float)

# Show new data types.
df_sb_yields.dtypes

In [ ]:
df_yields_sb_long